In [10]:
import tensorflow as tf
print(tf.keras)
print(tf)
print(tf.__file__)
from transformers import BertTokenizer, BertModel
import pandas as pd
import re

<KerasLazyLoader>
<module 'tensorflow' from 'c:\\Users\\donof\\AppData\\Local\\Programs\\Python\\Python310\\lib\\site-packages\\tensorflow\\__init__.py'>
c:\Users\donof\AppData\Local\Programs\Python\Python310\lib\site-packages\tensorflow\__init__.py


In [11]:
def preprocessText(text):
    text = text.lower() # lowercase text
    text = re.sub(r"(.|?)$", "", text) # removing trailing period/question mark from text
    text = re.sub(r"[^A-Za-z\s]{1,}'[A-Za-z]{1,}|[^A-Za-z0-9]{1,}\-[A-Za-z0-9]{1,}|[^A-za-z0-9]{1,}\.[A-za-z0-9]{0,}|[^A-Za-z0-9]{1,}\/[A-Za-z0-9]{1,}|^(#[.]{1,})","", text)
    return text

In [12]:
def normalizeDataset(path_to_dataset):
    dataset = pd.read_csv(path_to_dataset) # read dataset from csv
    
    context, response = dataset['Context'].values, dataset['Response'].values

    for i, text in enumerate(context):
        context[i] = preprocessText(text)
    
    for i, text in enumerate(response):
        response[i] = preprocessText(text)
    
    

In [13]:
def getEmbeddings(input):
    print(f"Prompt: {input}")
    
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased") # tokenizer
    model = BertModel.from_pretrained("bert-base-uncased", use_safetensors=False) # creating BERT model
    encoded_input = tokenizer(input, return_tensors='pt') # encode input
    return model(**encoded_input) # return features

In [14]:
def bilstm(embeddings):
    
    model = tf.keras.Sequential([
        embeddings,
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(128, return_sequences=True)),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(128)),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

In [15]:
# normalizeDataset("Dataset.csv")

In [16]:
embeddings = getEmbeddings("I feel down")
embeddings

Prompt: I feel down


c:\Users\donof\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[-0.1197,  0.5621,  0.1787,  ..., -0.1554,  0.1019,  0.1344],
         [-0.1263,  0.6111,  0.3071,  ..., -0.3046,  0.5088,  0.0117],
         [-0.1713,  0.4111,  0.5560,  ...,  0.0044,  0.0822,  0.4020],
         [-0.5414,  0.4621,  0.3132,  ...,  0.4774,  0.3280,  0.3281],
         [ 0.7758,  0.3602, -0.3593,  ..., -0.0510, -0.6552, -0.3917]]],
       grad_fn=<NativeLayerNormBackward0>), pooler_output=tensor([[-0.8263, -0.1162,  0.6851,  0.6289, -0.4345, -0.0957,  0.8209,  0.1625,
          0.3708, -0.9994,  0.4391,  0.0230,  0.9718, -0.3055,  0.9120, -0.4817,
         -0.1478, -0.4648,  0.2829, -0.7720,  0.4694,  0.4742,  0.6257,  0.1445,
          0.2911, -0.0414, -0.4264,  0.9099,  0.9374,  0.6510, -0.6693,  0.1175,
         -0.9746, -0.1052,  0.5651, -0.9643,  0.0184, -0.7007,  0.0556,  0.1272,
         -0.8659,  0.1392,  0.9872, -0.2686, -0.1850, -0.2203, -0.9982,  0.1978,
         -0.8405, -0.6214, -0.4909, 

In [17]:
bilstm(embeddings)

TypeError: The added layer must be an instance of class Layer. Received: layer=BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[-0.1197,  0.5621,  0.1787,  ..., -0.1554,  0.1019,  0.1344],
         [-0.1263,  0.6111,  0.3071,  ..., -0.3046,  0.5088,  0.0117],
         [-0.1713,  0.4111,  0.5560,  ...,  0.0044,  0.0822,  0.4020],
         [-0.5414,  0.4621,  0.3132,  ...,  0.4774,  0.3280,  0.3281],
         [ 0.7758,  0.3602, -0.3593,  ..., -0.0510, -0.6552, -0.3917]]],
       grad_fn=<NativeLayerNormBackward0>), pooler_output=tensor([[-0.8263, -0.1162,  0.6851,  0.6289, -0.4345, -0.0957,  0.8209,  0.1625,
          0.3708, -0.9994,  0.4391,  0.0230,  0.9718, -0.3055,  0.9120, -0.4817,
         -0.1478, -0.4648,  0.2829, -0.7720,  0.4694,  0.4742,  0.6257,  0.1445,
          0.2911, -0.0414, -0.4264,  0.9099,  0.9374,  0.6510, -0.6693,  0.1175,
         -0.9746, -0.1052,  0.5651, -0.9643,  0.0184, -0.7007,  0.0556,  0.1272,
         -0.8659,  0.1392,  0.9872, -0